In [3]:
# Базовые библиотеки для воспроизводимости, работы с данными и удобного вывода результатов.
import os
import sys
import torch
import random
import subprocess
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib 

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def ensure_package(package_name: str, import_name: Optional[str] = None) -> None:
    """Пытается импортировать пакет и при необходимости установить его через pip."""
    target = import_name or package_name
    try:
        __import__(target)
    except Exception:
        print(f"Устанавливаем пакет: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


# Для retrieval-контура попробуем установить основные зависимости.
# Даже если sentence-transformers не поднимется, ноутбук сможет работать через fallback.
ensure_package("faiss-cpu", "faiss")
ensure_package("sentence-transformers", "sentence_transformers")


try:
    import faiss  # type: ignore
    FAISS_AVAILABLE = True
except Exception as e:
    FAISS_AVAILABLE = False
    print("FAISS недоступен, будет использован fallback на sklearn NearestNeighbors.")
    print("Причина:", repr(e))


print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS available:", FAISS_AVAILABLE)

NumPy: 2.3.5
Pandas: 2.3.3
FAISS available: True


In [11]:
# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("Устройство для работы:", DEVICE)

Устройство для работы: cpu


Я возьму набор данных из Dataset: BBCSport, в нем 5 папок по темам:athletics, cricket, football, rugby, tennis. Я взял 30 статей из football.

In [14]:
folder_path = "bbcsport-fulltext/bbcsport/football"

# Загружаем 30 статей
files = sorted([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])[:30]

articles = []
for file in files:
    with open(os.path.join(folder_path, file), encoding="utf-8") as f:
        articles.append(f.read())

# Выводим первые 5
for i in range(5):
    print(articles[i][:500])
    print("\n" + "="*50 + "\n")

print(f"Загружено {len(articles)} статей")

Man Utd stroll to Cup win

Wayne Rooney made a winning return to Everton as Manchester United cruised into the FA Cup quarter-finals.

Rooney received a hostile reception, but goals in each half from Quinton Fortune and Cristiano Ronaldo silenced the jeers at Goodison Park. Fortune headed home after 23 minutes before Ronaldo scored when Nigel Martyn parried Paul Scholes' free-kick. Marcus Bent missed Everton's best chance when Roy Carroll, who was later struck by a missile, saved at his feet.

R


Van Nistelrooy set to return

Manchester United striker Ruud van Nistelrooy may make his comeback after an Achilles tendon injury in the FA Cup fifth round tie at Everton on Saturday.

He has been out of action for nearly three months and had targeted a return in the Champions League tie with AC Milan on 23 February. But Manchester United manager Sir Alex Ferguson hinted he may be back early. He said: "There is a chance he could be involved at Everton but we'll just have to see how he comes t

База знаний состоит из новостных статей о футболе, каждая из которых содержит факты, события, команды, игроков и результаты матчей. Это узкая и понятная предметная область, где тексты достаточно содержательны для формулирования вопросов. Такой корпус идеально подходит для retrieval или mini-RAG, поскольку позволяет быстро находить релевантные фрагменты и строить ответы на конкретные вопросы по теме.

In [15]:
df = pd.DataFrame(articles, columns=['text'])
df['doc_id'] = df.index
df

,text,doc_id
0,Man Utd stroll to Cup win\n\nWayne Rooney made...,0
1,Van Nistelrooy set to return\n\nManchester Uni...,1
2,Moyes U-turn on Beattie dismissal\n\nEverton m...,2
3,Ronaldo considering new contract\n\nManchester...,3
4,Smith keen on Home series return\n\nScotland m...,4
5,Mido makes third apology\n\nAhmed 'Mido' Hossa...,5
6,Man City 0-2 Man Utd\n\nManchester United redu...,6
7,Gerrard plays down European hopes\n\nSteven Ge...,7
8,Duff ruled out of Barcelona clash\n\nChelsea's...,8
9,Chelsea clinch cup in extra-time\n\n(after ext...,9


Чанкинг документов

In [16]:
# Простая функция чанкинга по словам.
def chunk_text(text: str, chunk_size: int = 22, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks

rows = []
for _, row in df.iterrows():
    chunks = chunk_text(row['text'], chunk_size=100, overlap=20)
    for chunk_id, chunk in enumerate(chunks):
        rows.append({
            'doc_id': row['doc_id'],
            'chunk_id': chunk_id,
            'chunk_text': chunk,
            'n_words': len(chunk.split())
        })

result_df = pd.DataFrame(rows)

result_df

,doc_id,chunk_id,chunk_text,n_words
0,0,0,Man Utd stroll to Cup win Wayne Rooney made a ...,100
1,0,1,feet. Rooney's return was always going to be a...,100
2,0,2,on a Goodison Park pitch that was cutting up. ...,100
3,0,3,"onside by Gabriel Heintze, hesitated and Carro...",100
4,0,4,their lead after 57 minutes as they doubled th...,100
...,...,...,...,...
175,28,2,has a hamstring injury. And Brown now looks ce...,61
176,29,0,Ferdinand casts doubt over Glazer Rio Ferdinan...,100
177,29,1,"bringing to the table."" The central defender a...",100
178,29,2,"to the stock exchange said: ""The board has not...",100


chunk_size = 100 — потому что тексты новостные, 100 слов вмещают законченную мысль (заголовок + несколько предложений) и не разрывают цитаты.( средняя длина статей 500-1500 слов)
overlap = 20 — потому что 20% перекрытия достаточно, чтобы смысл плавно перетекал из чанка в чанк, но не создаёт слишком много дублирования.
По итогу параметры подобраны так, чтобы чанк вмещал 3-5 предложений (целую цитату или смысловой фрагмент), а перекрытие в 20 слов обеспечивало связность между чанками без потери контекста.

Эмбеддинги и индекс FAISS

In [17]:
# Единый интерфейс для двух вариантов векторизации: dense embeddings и fallback.
class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore

        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        return vectors.astype("float32")


class TfidfFallbackBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
        self.backend_name = "TF-IDF fallback"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.fit_transform(texts).toarray()
        return vectors.astype("float32")

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.vectorizer.transform(texts).toarray()
        return vectors.astype("float32")


def build_embedding_backend(
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    device: str = "cpu",
) -> EmbeddingBackend:
    try:
        backend = SentenceTransformersBackend(model_name=model_name, device=device)
        print("Используем полноценные dense embeddings.")
        print("Бэкэнд:", backend.backend_name)
        return backend
    except Exception as e:
        print("Не удалось загрузить sentence-transformers encoder.")
        print("Причина:", repr(e))
        print("Переключаемся на TF-IDF fallback. Ноутбук останется рабочим,")
        print("но это уже не полноценные dense embeddings.")
        return TfidfFallbackBackend()


embedder = build_embedding_backend(device=DEVICE)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3500.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Используем полноценные dense embeddings.
Бэкэнд: SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


In [18]:
# Строим векторные представления для всех чанков.
chunk_texts = result_df["chunk_text"].tolist()
chunk_embeddings = embedder.fit_documents(chunk_texts)

print("Форма матрицы эмбеддингов:", chunk_embeddings.shape)

# Проверяем длины векторов.
# Если normalize_embeddings=True сработал корректно, все нормы должны быть ≈ 1.0.
# Это означает, что косинусное сходство далее можно считать через скалярное произведение.
vector_norms = np.linalg.norm(chunk_embeddings, axis=1)
print("Минимальная норма:", round(float(vector_norms.min()), 4))
print("Максимальная норма:", round(float(vector_norms.max()), 4))
print("Средняя норма:", round(float(vector_norms.mean()), 4))
print("→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.")

Форма матрицы эмбеддингов: (180, 384)
Минимальная норма: 1.0
Максимальная норма: 1.0
Средняя норма: 1.0
→ Нормы ≈ 1.0: нормировка подтверждена, dot product = cosine similarity.


In [19]:
# Единая обёртка над FAISS и fallback-поиском.
class VectorSearchIndex:
    def __init__(self, dim: int) -> None:
        self.dim = dim
        self.backend_name = None
        self._faiss_index = None
        self._nn_index = None

        if FAISS_AVAILABLE:
            self._faiss_index = faiss.IndexFlatIP(dim)  # type: ignore[name-defined]
            self.backend_name = "FAISS IndexFlatIP"
        else:
            self._nn_index = NearestNeighbors(metric="cosine")
            self.backend_name = "sklearn NearestNeighbors fallback"

    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")

        if self._faiss_index is not None:
            self._faiss_index.add(vectors)
        else:
            self._nn_index.fit(vectors)

    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")

        if self._faiss_index is not None:
            scores, indices = self._faiss_index.search(query_vectors, top_k)
            return scores, indices

        distances, indices = self._nn_index.kneighbors(query_vectors, n_neighbors=top_k)
        scores = 1.0 - distances
        return scores, indices


search_index = VectorSearchIndex(dim=chunk_embeddings.shape[1])
search_index.add(chunk_embeddings)

print("Индекс построен.")
print("Бэкэнд индекса:", search_index.backend_name)

Индекс построен.
Бэкэнд индекса: FAISS IndexFlatIP


In [20]:
# Удобная функция для поиска похожих чанков.
def search_similar_chunks(query: str, chunks_df: pd.DataFrame, search_index: VectorSearchIndex, top_k: int = 5) -> pd.DataFrame:
    query_vectors = embedder.encode_queries([query])
    scores, indices = search_index.search(query_vectors, top_k=top_k)

    rows = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        chunk_row = chunks_df.iloc[int(idx)]
        rows.append(
            {
                "rank": rank,
                "doc_id": chunk_row["doc_id"],
                "chunk_id": int(chunk_row["chunk_id"]),
                "score": round(float(score), 4),
                "chunk_text": chunk_row["chunk_text"],
            }
        )

    return pd.DataFrame(rows)

In [152]:
# Проверяем retrieval на нескольких запросах.
example_queries = [
    "How did Manchester United beat Everton in the FA Cup?",
    "When will Ruud van Nistelrooy return from his Achilles injury?",
    "Why did David Moyes change his mind about James Beattie's red card for headbutting?",
    "What did Cristiano Ronaldo say about his new contract at Manchester United?",
    "Why does Walter Smith want to bring back the Home International series?",
]

for current_query in example_queries:
    display(Markdown(f"### Запрос: `{current_query}`"))
    display(search_similar_chunks(current_query, result_df,search_index,top_k=5))

### Запрос: `How did Manchester United beat Everton in the FA Cup?`

,rank,doc_id,chunk_id,score,chunk_text
0,1,6,6,0.5819,"one Wayne Rooney scored from."" - Manchester Un..."
1,2,0,0,0.5704,Man Utd stroll to Cup win Wayne Rooney made a ...
2,3,0,4,0.5333,their lead after 57 minutes as they doubled th...
3,4,0,3,0.5295,"onside by Gabriel Heintze, hesitated and Carro..."
4,5,6,5,0.5230,a had a third late on when substitute Ryan Gig...


### Запрос: `When will Ruud van Nistelrooy return from his Achilles injury?`

,rank,doc_id,chunk_id,score,chunk_text
0,1,1,0,0.7671,Van Nistelrooy set to return Manchester United...
1,2,1,1,0.6200,at Everton but we'll just have to see how he c...
2,3,19,1,0.5704,because of the swelling it was impossible to m...
3,4,15,1,0.5686,"Barcelona game."" His comments contradict those..."
4,5,1,2,0.5500,they will be boosted by the return of the Dutc...


### Запрос: `Why did David Moyes change his mind about James Beattie's red card for headbutting?`

,rank,doc_id,chunk_id,score,chunk_text
0,1,2,1,0.6548,"Moyes added: ""My comments on Saturday came imm..."
1,2,2,0,0.6434,Moyes U-turn on Beattie dismissal Everton mana...
2,3,2,2,0.5908,in his career - his actions were unacceptable ...
3,4,23,9,0.4795,"""an eye-opener"" and admits Liverpool would be ..."
4,5,2,3,0.4699,do still believe the Chelsea player in questio...


### Запрос: `What did Cristiano Ronaldo say about his new contract at Manchester United?`

,rank,doc_id,chunk_id,score,chunk_text
0,1,3,0,0.7608,Ronaldo considering new contract Manchester Un...
1,2,3,1,0.6130,I think we'll reach a good agreement for both ...
2,3,17,7,0.5222,and then tell him 'by the way we've decided to...
3,4,16,5,0.5005,"each other. ""By all accounts it's pretty serio..."
4,5,20,6,0.4794,"be 35 so hopefully I will still be okay. ""A lo..."


### Запрос: `Why does Walter Smith want to bring back the Home International series?`

,rank,doc_id,chunk_id,score,chunk_text
0,1,4,0,0.6647,Smith keen on Home series return Scotland mana...
1,2,4,1,0.4080,friendly games and that's something that's nee...
2,3,24,3,0.3725,manager has decided to have a training camp in...
3,4,17,12,0.3566,"didn't happen. ""Quite genuinely, the new inter..."
4,5,26,12,0.3506,"it didn't happen. ""Quite genuinely, the new in..."


Контрольные запросы и оценка retrieval

In [22]:
benchmark_queries: List[Dict[str, object]] = [
    {
        "query_id": "q01",
        "query": "How did Manchester United beat Everton in the FA Cup?",
        "relevant_doc_ids": [0],
    },
    {
        "query_id": "q02",
        "query": "When will Ruud van Nistelrooy return from his Achilles injury?",
        "relevant_doc_ids": [1],
    },
    {
        "query_id": "q03",
        "query": "Why did David Moyes change his mind about James Beattie's red card for headbutting?",
        "relevant_doc_ids": [2],
    },
    {
        "query_id": "q04",
        "query": "What did Cristiano Ronaldo say about his new contract at Manchester United?",
        "relevant_doc_ids": [3],
    },
    {
        "query_id": "q05",
        "query": "Why does Walter Smith want to bring back the Home International series?",
        "relevant_doc_ids": [4],
    },
    {
        "query_id": "q06",
        "query": "Why did Mido apologize to the Egyptian national team?",
        "relevant_doc_ids": [5],
    },
    {
        "query_id": "q07",
        "query": "How did the Manchester derby between Manchester City and Manchester United end?",
        "relevant_doc_ids": [6],
    },
    {
        "query_id": "q08",
        "query": "Why doesn't Steven Gerrard believe Liverpool can win the Champions League?",
        "relevant_doc_ids": [7],
    },
    {
        "query_id": "q09",
        "query": "How did Chelsea win the Carling Cup final against Liverpool in extra time?",
        "relevant_doc_ids": [9],
    },
    {
        "query_id": "q10",
        "query": "How did Newcastle United beat Bolton Wanderers 2-1?",
        "relevant_doc_ids": [10],
    },
    {
        "query_id": "q11",
        "query": "How did Middlesbrough draw 2-2 with Charlton Athletic?",
        "relevant_doc_ids": [11],
    },
    {
        "query_id": "q12",
        "query": "How did Dundee United beat Aberdeen 4-1 in the Scottish Cup?",
        "relevant_doc_ids": [12],
    },
    {
        "query_id": "q13",
        "query": "How did Celtic beat Clyde 5-0 in the Scottish Cup?",
        "relevant_doc_ids": [13],
    },
    {
        "query_id": "q14",
        "query": "How did Chelsea's Arjen Robben get injured and how long will he be out?",
        "relevant_doc_ids": [19],
    },
    {
        "query_id": "q15",
        "query": "What did Rio Ferdinand say about Malcolm Glazer's bid to buy Manchester United?",
        "relevant_doc_ids": [29],
    },
]

In [21]:
def evaluate_query(
    query: str,
    relevant_doc_ids: List[int],
    chunks_df: pd.DataFrame,
    embedder: EmbeddingBackend,
    search_index: VectorSearchIndex,
    top_k: int = 3,
) -> Dict[str, object]:
    result_df = search_similar_chunks(query, chunks_df, search_index, top_k=top_k)  # ← добавить search_index
    
    predicted_doc_ids = []
    seen = set()
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            predicted_doc_ids.append(doc_id)
    
    hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))
    recall = sum(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids) / len(relevant_doc_ids)
    
    first_relevant_rank = None
    for idx, doc_id in enumerate(predicted_doc_ids, start=1):
        if doc_id in relevant_doc_ids:
            first_relevant_rank = idx
            break
    
    mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank
    
    return {
        "predicted_doc_ids": predicted_doc_ids,
        "hit@k": hit,
        "recall@k": recall,
        "first_relevant_rank": first_relevant_rank,
        "mrr": mrr,
        "result_df": result_df,
    }
  


def evaluate_benchmark(
    benchmark_rows: List[Dict[str, object]],
    chunks_df: pd.DataFrame,
    embedder: EmbeddingBackend,
    search_index: VectorSearchIndex,
    top_k: int = 3,
) -> pd.DataFrame:
    rows = []
    for row in benchmark_rows:
        metrics = evaluate_query(
            query=row["query"],
            relevant_doc_ids=row["relevant_doc_ids"],
            chunks_df=chunks_df,
            embedder=embedder,
            search_index=search_index,
            top_k=top_k,
        )
        rows.append(
            {
                "query_id": row["query_id"],
                "query": row["query"],
                "expected_source": ", ".join(str(x) for x in row["relevant_doc_ids"]),
                "retrieved_sources": ", ".join(str(x) for x in metrics["predicted_doc_ids"]),
                f"hit_at_k": metrics["hit@k"],
                f"recall_at_k": round(metrics["recall@k"], 4),
                f"MRR_at_k": round(metrics["mrr"], 4),
                "rank_of_first_relevant": metrics["first_relevant_rank"],
            }
        )
    return pd.DataFrame(rows)

In [23]:
results = evaluate_benchmark(benchmark_queries, result_df, embedder, search_index, top_k=3)
results.to_csv("artifacts/retrieval_eval.csv", index=False, encoding="utf-8")
display(results)

,query_id,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,MRR_at_k,rank_of_first_relevant
0,q01,How did Manchester United beat Everton in the ...,0,"6, 0",1,1.0,0.5,2
1,q02,When will Ruud van Nistelrooy return from his ...,1,"1, 19",1,1.0,1.0,1
2,q03,Why did David Moyes change his mind about Jame...,2,2,1,1.0,1.0,1
3,q04,What did Cristiano Ronaldo say about his new c...,3,"3, 17",1,1.0,1.0,1
4,q05,Why does Walter Smith want to bring back the H...,4,"4, 24",1,1.0,1.0,1
5,q06,Why did Mido apologize to the Egyptian nationa...,5,5,1,1.0,1.0,1
6,q07,How did the Manchester derby between Mancheste...,6,6,1,1.0,1.0,1
7,q08,Why doesn't Steven Gerrard believe Liverpool c...,7,"7, 17",1,1.0,1.0,1
8,q09,How did Chelsea win the Carling Cup final agai...,9,"9, 7",1,1.0,1.0,1
9,q10,How did Newcastle United beat Bolton Wanderers...,10,10,1,1.0,1.0,1


Небольшой эксперимент с параметрами retrieval (сравню два значения top_k)

In [157]:
topk_rows = []

for top_k in [1,5]:
    eval_df = evaluate_benchmark(benchmark_queries, result_df, embedder, search_index, top_k=top_k)
    topk_rows.append(
        {
            "top_k": top_k,
            "mean_hit": eval_df[f"hit@{top_k}"].mean(),
            "mean_recall": eval_df[f"recall@{top_k}"].mean(),
            "mean_MRR": eval_df[f"MRR@{top_k}"].mean(),
        }
    )

topk_df = pd.DataFrame(topk_rows)
display(topk_df)

,top_k,mean_hit,mean_recall,mean_MRR
0,1,0.933333,0.933333,0.933333
1,5,1.000000,1.000000,0.966667


Обновление базы знаний и переиндексация

In [24]:
# Загружаем новые 5 статей
files_new = sorted([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])[:35]

articles_new = []
for file in files_new:
    with open(os.path.join(folder_path, file), encoding="utf-8") as f:
        articles_new.append(f.read())

print(f"Загружено {len(articles_new)} статей")

Загружено 35 статей


In [25]:
#переиндексация
df_new = pd.DataFrame(articles_new, columns=['text'])
df_new['doc_id'] = df_new.index
#повторный чанкинг
rows_new = []
for _, row_new in df_new.iterrows():
    chunks_new = chunk_text(row_new['text'], chunk_size=100, overlap=20)
    for chunk_id_new, chunk_new in enumerate(chunks_new):
        rows_new.append({
            'doc_id': row_new['doc_id'],
            'chunk_id': chunk_id_new,
            'chunk_text': chunk_new,
            'n_words': len(chunk_new.split())
        })

result_df_new = pd.DataFrame(rows_new)

chunk_texts_new = result_df_new["chunk_text"].tolist()
chunk_embeddings_new = embedder.fit_documents(chunk_texts_new)

search_index_new = VectorSearchIndex(dim=chunk_embeddings_new.shape[1])
search_index_new.add(chunk_embeddings_new)


# Новые запросы
new_queries = [
    {"query_id": "q16", "query": "What are the allegations about Chelsea making an illegal approach for Ashley Cole?", "relevant_doc_ids": [30]},
    {"query_id": "q17", "query": "Why did Dundee United manager Ian McCall get a reprieve from being sacked?", "relevant_doc_ids": [31]},
    {"query_id": "q18", "query": "Why is Harry Kewell's return from injury taking so long?", "relevant_doc_ids": [32]},
    {"query_id": "q19", "query": "What does Walter Smith want to achieve as the new Scotland manager?", "relevant_doc_ids": [33]},
    {"query_id": "q20", "query": "Which Wales players are injured before the friendly against Hungary?", "relevant_doc_ids": [34]},
]

# Объединяем 
all_queries = benchmark_queries + new_queries

results_new = evaluate_benchmark(all_queries, result_df_new, embedder, search_index_new, top_k=3)
display(results_new)

,query_id,query,expected_source,retrieved_sources,hit_at_k,recall_at_k,MRR_at_k,rank_of_first_relevant
0,q01,How did Manchester United beat Everton in the ...,0,"6, 0",1,1.0,0.5,2
1,q02,When will Ruud van Nistelrooy return from his ...,1,"1, 19",1,1.0,1.0,1
2,q03,Why did David Moyes change his mind about Jame...,2,2,1,1.0,1.0,1
3,q04,What did Cristiano Ronaldo say about his new c...,3,"3, 17",1,1.0,1.0,1
4,q05,Why does Walter Smith want to bring back the H...,4,"4, 33",1,1.0,1.0,1
5,q06,Why did Mido apologize to the Egyptian nationa...,5,5,1,1.0,1.0,1
6,q07,How did the Manchester derby between Mancheste...,6,6,1,1.0,1.0,1
7,q08,Why doesn't Steven Gerrard believe Liverpool c...,7,"7, 17",1,1.0,1.0,1
8,q09,How did Chelsea win the Carling Cup final agai...,9,"9, 7",1,1.0,1.0,1
9,q10,How did Newcastle United beat Bolton Wanderers...,10,10,1,1.0,1.0,1


In [ ]:
# Переименовываем колонки для старых результатов
results_old_renamed = results.copy()
results_old_renamed.columns = [f"before_{col}" if col not in ['query_id', 'query'] else col for col in results_old_renamed.columns]

# Переименовываем колонки для новых результатов
results_new_renamed = results_new.copy()
results_new_renamed.columns = [f"after_{col}" if col not in ['query_id', 'query'] else col for col in results_new_renamed.columns]


detailed_comparison = pd.merge(
    results_old_renamed, 
    results_new_renamed, 
    on=['query_id', 'query'], 
    how='outer'
)

# Добавляем поле changed для retrieved_sources_doc_ids
detailed_comparison['changed'] = (
    detailed_comparison['before_retrieved_sources'].fillna('') != 
    detailed_comparison['after_retrieved_sources'].fillna('')
)

detailed_comparison.to_csv("artifacts/retrieval_before_after_update.csv", index=False, encoding="utf-8")
display(detailed_comparison)

Видны изменения для 5 и 12 вопросов!

Mini-RAG

In [165]:
def build_context_from_retrieval(query: str, chunks_df: pd.DataFrame, search_index: VectorSearchIndex, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = search_similar_chunks(query, chunks_df, search_index, top_k=top_k)
    context_blocks = []

    for _, row in retrieved.iterrows():
        block = (
            f"[Источник: doc_id={row['doc_id']} | chunk_id={row['chunk_id']} | score={row['score']:.4f}]\n"
            f"{row['chunk_text']}"
        )
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)
    return context, retrieved

def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    # Убираем технические строки источников из ранжирования, но не из общего контекста.
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    content_lines = [line for line in raw_lines if not line.startswith("[Источник:")]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))

    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 4]

    if not sentence_pool:
        return "Недостаточно контекста для построения ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec = matrix[0]
    sentence_vecs = matrix[1:]

    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used_normalized = set()

    for idx in ranked_idx:
        sentence = sentence_pool[idx]
        normalized = sentence.lower().strip()
        if scores[idx] <= 0:
            continue
        if normalized in used_normalized:
            continue
        used_normalized.add(normalized)
        selected_sentences.append(sentence)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа."

    return " ".join(selected_sentences)

def mini_rag_answer(
    query: str,
    chunks_df: pd.DataFrame,
    search_index: VectorSearchIndex,
    top_k: int = 3,
    max_answer_sentences: int = 2,
) -> Dict[str, object]:
    context, retrieved = build_context_from_retrieval(query, chunks_df, search_index, top_k=top_k)
    answer = generate_answer_from_context(query, context=context, max_sentences=max_answer_sentences)

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "sources": retrieved,
    }

In [191]:
comparison_queries = [
    "How did Manchester United beat Everton in the FA Cup?",
    "When will Ruud van Nistelrooy return from his Achilles injury?",
    "Why did David Moyes change his mind about James Beattie's red card for headbutting?",
    "What did Cristiano Ronaldo say about his new contract at Manchester United?",
    "Why does Walter Smith want to bring back the Home International series?",
    "Why did Mido apologize to the Egyptian national team?",
    "How did the Manchester derby between Manchester City and Manchester United end?",
    "Why doesn't Steven Gerrard believe Liverpool can win the Champions League?",
    "How did Chelsea win the Carling Cup final against Liverpool in extra time?",
    "How did Newcastle United beat Bolton Wanderers 2-1?",
    "How did Middlesbrough draw 2-2 with Charlton Athletic?",
    "How did Dundee United beat Aberdeen 4-1 in the Scottish Cup?",
    "How did Celtic beat Clyde 5-0 in the Scottish Cup?",
    "How did Chelsea's Arjen Robben get injured and how long will he be out?",
    "What did Rio Ferdinand say about Malcolm Glazer's bid to buy Manchester United?",
]

# Собираем данные для CSV
rag_rows = []

for _, row in comparison_df.iterrows():
    rag_result = mini_rag_answer(
        query=row['query'],
        chunks_df=result_df,      
        search_index=search_index, 
        top_k=3
    )
    
    # Формируем retrieved_sources в читаемом виде
    sources_str = "; ".join([
        f"doc_{row['doc_id']} (score={row['score']:.3f})"
        for _, row in rag_result["sources"].iterrows()
    ])
    
    rag_rows.append({
        "question": rag_result["query"],
        "answer": rag_result["answer"],
        "retrieved_sources": sources_str
    })
    
    # Выводим на экран
    display(Markdown(f"## Вопрос: {rag_result['query']}"))
    display(Markdown(f"**Mini-RAG:** {rag_result['answer']}"))
    display(rag_result["sources"][["rank", "score", "doc_id", "chunk_text"]])

# Сохраняем в CSV
rag_df = pd.DataFrame(rag_rows)
rag_df.to_csv("artifacts/rag_examples.csv", index=False, encoding="utf-8")

## Вопрос: How did Manchester United beat Everton in the FA Cup?

**Mini-RAG:** Man Utd stroll to Cup win Wayne Rooney made a winning return to Everton as Manchester United cruised into the FA Cup quarter-finals. "At times, especially in the first half, we didn't play with enough speed.

,rank,score,doc_id,chunk_text
0,1,0.5819,6,"one Wayne Rooney scored from."" - Manchester Un..."
1,2,0.5704,0,Man Utd stroll to Cup win Wayne Rooney made a ...
2,3,0.5333,0,their lead after 57 minutes as they doubled th...


## Вопрос: When will Ruud van Nistelrooy return from his Achilles injury?

**Mini-RAG:** Van Nistelrooy set to return Manchester United striker Ruud van Nistelrooy may make his comeback after an Achilles tendon injury in the FA Cup fifth round tie at Everton on Saturday. And he added: "It felt different then last summer when I had the same injury on my other foot.

...

**Mini-RAG:** Robben sidelined with broken foot Chelsea winger Arjen Robben has broken two metatarsal bones in his foot and will be out for at least six weeks. Kenyon denies Robben Barca return Chelsea chief executive Peter Kenyon has played down reports that Arjen Robben will return for the Champions League match against Barcelona.

,rank,score,doc_id,chunk_text
0,1,0.7577,19,because of the swelling it was impossible to m...
1,2,0.7531,19,Robben sidelined with broken foot Chelsea wing...
2,3,0.6496,15,Kenyon denies Robben Barca return Chelsea chie...


## Вопрос: What did Rio Ferdinand say about Malcolm Glazer's bid to buy Manchester United?

**Mini-RAG:** Ferdinand casts doubt over Glazer Rio Ferdinand has said he is unsure of Malcolm Glazer's motives after the American billionaire launched a new offer to buy Manchester United. "The board can confirm that it has now received a detailed proposal subject to various pre-conditions which may form the basis of an offer for Manchester United from Glazer.

,rank,score,doc_id,chunk_text
0,1,0.7863,29,Ferdinand casts doubt over Glazer Rio Ferdinan...
1,2,0.6164,29,"bringing to the table."" The central defender a..."
2,3,0.6090,29,"to the stock exchange said: ""The board has not..."


Краткий анализ ошибок
1. Вопрос про Роббена (Arjen Robben):ответ начинается правильно (про травму), но затем добавляет фрагмент из другой статьи ("Kenyon denies Robben Barca return"), где обсуждается возможный возврат, а не травма.
Ретривал вернул doc_19 (правильно) и doc_15 (тоже про Роббена, но по другой теме). Контекст получился смешанным, и генератор ответа взял предложения из обоих источников.Получается проблема в составе контекста — в топ-3 попали две разные статьи про Роббена, но с разными сюжетами. 
2. Вопрос про Дэвида Мойеса (Beattie red card):ответ обрывается на полуслове: "James did issue a".Генератор ответа выбрал предложение, которое оказалось неполным или было обрезано при чанкинге.Это уже говорит о качестве чанкинга — разрыв предложения на границе чанка.
3. Вопрос про Манчестерское дерби:ответ содержит цитату Кина, которая не отвечает прямо на вопрос "как закончился матч".Ретривал вернул чанки, где есть цитаты тренеров, но не описание гола. Вопрос сформулирован как "how did it end", а модель предпочла цитату фактическому описанию.
Тут проблема может быть в формулировке вопроса — можно было уточнить "what was the score".